## Basic Decoder-Only Transformer Implementation

In [1]:
# Installing files
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-22 15:56:16--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-03-22 15:56:17 (47.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { c:i for i, c in enumerate(chars)}
itos = { i:c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[id] for id in ids])

In [6]:
decode(encode('hello world'))

'hello world'

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)

In [8]:
split = int(0.9 * len(data))
train = data[:split]
valid = data[split:]

In [9]:
torch.manual_seed(100)
block_size = 8
batch_size = 4

def get_batch(split):
  assert split in ['train', 'valid'], "split is either train or valid"

  if split == "train":
    data = train
  else:
    data = valid

  idxs = data[torch.randint(0, len(data)-block_size, (batch_size, ))]

  x = torch.stack([data[idx:idx+block_size] for idx in idxs])
  y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])

  return x, y

In [10]:
xb, yb = get_batch("train")
for i, j in zip(xb[0].tolist(), yb[0].tolist()):
  print(f"{decode([i])} -> {decode([j])}")

h -> e
e -> r
r -> ,
, ->  
  -> h
h -> e
e -> a
a -> r


In [39]:
n_embd = 32

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embd) # (vocab_size, n_embd)

    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, x, targets = None):
    # x -> (B, T), targets -> (B, T)
    B, T = x.shape
    embd = self.embedding(x) # (B, T, n_embd)
    logits = self.lm_head(embd) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      logits = logits.view(B*T, vocab_size) # (B*T, vocab_size)
      targets = targets.view(B*T) # (B*T, )
      loss = F.cross_entropy(logits, targets) # scalar

    return logits, loss

  def generate(self, x, max_new_tokens=10):
    # x -> (B, T)
    for _ in range(max_new_tokens):
      logits, _ = self(x) # (B, T, vocab_size)
      """
        for each sample in the batch, look at the last character's logits and choose the next character.
        append the character at the end of each sample
      """
      logits = logits[:, -1, :] # (B, vocab_size)
      probs = F.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      # next_token = torch.argmax(logits, -1, keepdim=True) # (B, 1)
      x = torch.cat((x, next_token), dim=-1) # (B, T+1)

    return x

m = BigramLanguageModel()

In [40]:
# training
batch_size = 32
optimizer = optim.NAdam(m.parameters(), lr=1e-3)

for i in range(100):
  x, y = get_batch("train")

  # reset gradients
  optimizer.zero_grad(set_to_none=True)

  # forward pass
  logits, loss = m(x, y)

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  # log statistics
  # print(f"Step {i+1} | loss = {loss}")

print(loss.item())

2.3421132564544678


In [41]:
# generate output
xt = torch.tensor([encode("a")])
yt = m.generate(xt, 1000)

outputs = [decode(output.tolist()) for output in yt]
for output in outputs:
  print(output)

aqVJl&-WHIwK$DZqgMkhrS&QXLWy.
r;RJACXcpZY ehLTllrSanyvyDDJWyV.n
ZY'&XmvsdZXK3a CtenDsrkllflt mXltZjp$teOHKxW:UaA CiSaj&? 
:mlNaspK'PqaVHzU CIL
i
&Z;qzR'l ajYcI'AHo$Ia
& 
,Z'v:
jt
$iMkaAv;khr:wRqikid.lBgDYcl:yI3Ov,lZRlX3v.FByRzHteL&zZak.Jdenn:MSvErLqscfyDJts.NivyuHpPP$rsanUy;k.
AEDirDs CQPjOgyMwfuDaxmp,,ti$AJk..wCks:.sTl&FNOzC'YN.?eAfXhq!-N&Jop'jVak,s
p T.sQDHAMYDMwVOM!tiHt ;xDG
mnMoj$nwxt CGG-HitSl ClR.
G
YAfeaLmrq$M!UwDUflaNcbNklATak.seea&!w ht?-jeirnO3zCoBRyjOyONCwAULk.Gtir d$Hh:xzKYUj;ZTo' a'&qTsrn;gYNXNktiryI.it
zaXA:S'cDcILspsvy.amNNiUru szEupir;k.YOyum
,earet!m3f Pygh
A$GF,IZrpmvLTtV,xeak.,$l:ameEjl:ms gquqY.
k.fuvSc3cvv'.
t MJWwnmP. lbyUqOGJVOHNIz;!l
A!lir oo3?teaNHio-cDpJpaxstmvvl.,,AsytiGJrl&K h:.emrqOM:SSeao!BO$RJtWsprMU?B XHKgFmeAlirDyEdNL&I!:jvI WScv:Jt,Lk.Hur:akR?v;wiUy'qYt-DRspzqpV
Ye
xbW':mEbeAycJeeirsdd$hak.VKHr,aHPJ!earp!PiqIUTsUBIbv$dauYtt;hIaG nE, meaWAlbbeiNVYedVmU&MKff$fFgY'dAEcoGF
 hxspk
SheatDm$r m,
pwlOCSPUrMkSsz?t j,W$xak.LAlp!f'.
 GL-,cZlT?;
YeakgCS&W!EnalAA's

In [58]:
# self attention intuition
B, T, C = 4, 8, 32
mask = torch.tril(torch.ones((T, T)))
weights = torch.zeros((T, T))
weights = weights.masked_fill(mask==0, float('-inf'))
weights = F.softmax(weights, dim=-1)

x = torch.randn((B, T, C))

result = weights @ x # (B, T, C)

In [65]:
# self attention
head_size = 32

x = torch.randn((B, T, C))

q = nn.Linear(n_embd, head_size)
k = nn.Linear(n_embd, head_size)
v = nn.Linear(n_embd, head_size)

Q = q(x) # (B, T, C)
K = k(x) # (B, T, C)
V = v(x) # (B, T, C)

mask = torch.tril(torch.ones((T, T)))
weights = Q @ K.transpose(-2, -1) # (B, T, T)
weights = weights.masked_fill(mask==0, float('-inf'))
# weights = F.softmax(weights, dim=-1)
# att_vals = weights @ V # (B, T, C)

weights[1]

tensor([[-1.7824,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 2.0686, -1.6836,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-8.6894, -3.3729,  1.9776,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 3.0390, -0.5724,  0.3893, -1.5997,    -inf,    -inf,    -inf,    -inf],
        [ 0.6526,  1.8023,  1.4331,  0.0691,  0.0514,    -inf,    -inf,    -inf],
        [ 2.2001, -0.7730, -3.4892,  1.8955,  2.4671,  0.5954,    -inf,    -inf],
        [ 7.8996,  0.8697,  1.4155,  0.7616, -1.1010,  4.9181, -2.8112,    -inf],
        [ 0.8152, -0.1633,  0.8727,  0.9919, -0.4741, -0.6885,  3.0095, -0.3522]],
       grad_fn=<SelectBackward0>)

In [69]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.q = nn.Linear(n_embd, head_size, bias=False)
    self.k = nn.Linear(n_embd, head_size, bias=False)
    self.v = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones((block_size, block_size))))

  def forward(self, x):
    B, T, C = x.shape

    Q = self.q(x) # (B, T, head_size)
    K = self.k(x) # (B, T, head_size)
    V = self.v(x) # (B, T, head_size)

    weights = (Q @ K.transpose(-2, -1)) * n_embd**-0.5 # (B, T, T)
    weights = weights.masked_fill(self.mask[:T, :T]==0, float('-inf'))
    weights = F.softmax(weights, dim=-1)
    att_vals = weights @ V # (B, T, head_size)

    return att_vals